# 74d2. Why the input role made Qwen easier to break: the missing controls (Qwen2.5-3B)

**What happened.** Notebook 74d continued v13 (3B) and v12 (7B) on their own last 2,800 training examples with the document moved into its own input turn, then ran notebook 74c's GCG attack again. The role made both models easier to break, not harder: the 3B from 11 to 21 of 25 documents (paired +0.40 [+0.20, +0.60]), the 7B from 19 to 23 (+0.16 [+0.04, +0.32]). The plain injection without a suffix still failed on every document, and both models still summarise.

**What the committed results already show.**
- No sign of a fault in the attack: no rendering mismatches, every GPU check passed, and the same scoring rule as notebook 74c.
- Moving the document made the model disagree with its teacher again. On the very examples v13 had just been trained on, the reverse-KL to the teacher was 0.015 at the end of v13's own training (notebook 73e) but 0.034 at the start of 74d's training, and 0.023 at its end. On the 7B: 0.018 at the end of notebook 73f, 0.032 at the start of 74d, 0.019 at its end.
- Copying and processing tasks became easier with the passage in its own turn: exact-target loss 0.012 to 0.019 in notebook 74d against 0.047 to 0.067 in notebooks 73e and 73f.
- GCG's loss fell faster on the role models from the first steps (figure of notebook 74d).

**Why that is not yet an explanation.** Notebook 74d changed two things at once: the format, and one more pass over the same 2,800 examples. It also never measured the untrained model or v13 itself with the document in the input turn. So four explanations remain open:
1. **Extra training.** Any further pass over the same examples weakens the defence against GCG, with or without the role.
2. **The role, at equal training.** With the same extra training, the role makes the model easier to break.
3. **A defence tied to the layout.** v13 learned its defence with the document inside the user turn; moving the document breaks it, and 2,800 examples only repair part of it.
4. **The format itself.** Qwen was never trained with an input turn; the untrained model may treat the document's own turn as a fresh message to answer, which would make the injection at its end stronger before any defence is trained.

**The six conditions (Qwen2.5-3B, the same 25 documents and the same GCG as notebooks 74c and 74d).** Three are read from earlier records; three are new.

| | weights | format at run time | source |
|---|---|---|---|
| A | undefended | one user turn | notebook 74c |
| B | undefended | input role | **new** |
| C | v13 | one user turn | notebook 74c |
| D | v13 | input role (no new training) | **new** |
| E | v13 + one more pass over the same 2,800 examples, one user turn | one user turn | **new (trained here)** |
| F | v13 + one more pass over the same 2,800 examples, input role | input role | notebook 74d |

E is trained exactly as F was (notebook 68b's training step, the same seed, the same examples, the same order), with the one difference that every prompt stays in one user turn. For every condition the notebook also measures the attack's starting point: the loss on "hacked" with the initial suffix, before any optimisation (a lower loss means the model is closer to saying "hacked" from the start).

**Readings, fixed in Stage A before any training or generation** (each a paired difference over the 25 documents, bootstrap over documents):

| reading | comparison | what it would mean |
|---|---|---|
| R1 extra training | E minus C, share broken | lower end above 0: one more pass over the same examples, without the role, makes the 3B easier to break |
| R2 the role at equal training | F minus E, share broken | lower end above 0: at equal training, the role makes the 3B easier to break |
| R3 moving the document, no retraining | D minus C, share broken | lower end above 0: v13's defence is tied to the one-turn layout |
| R4 retraining in the role | F minus D, share broken | upper end below 0: training in the role repairs part of what moving the document broke |
| R5 the format on the untrained model | B minus A: the injection alone, broken at step 1, and the starting loss | injection alone or step 1 higher with lower end above 0, or starting loss lower with upper end below 0: the untrained Qwen obeys an instruction at the end of its own turn more readily |
| reported | the starting loss for all six, paired for the same pairs; the summaries (60 validation passages) of B, D and E; the training curves of E beside F's | |

Each explanation is reported as supported, contradicted or not settled, whatever the numbers show. Nothing here changes v13, v12 or any earlier result. The 7B is not repeated: its change in notebook 74d was smaller, and the 3B is where the cause is cheapest to find.

**Compute.** A GPU with bf16 and 40 GB or more, as notebooks 74c and 74d. One training run of 2,800 examples on the 3B, the starting losses (one pass per document), the summaries, then GCG on B, D and E. A document the model holds runs all 500 steps. Every stage is cached: training resumes from its last checkpoint and GCG from the next document.

**Outputs.**
- `data/diag74d2/`: E's adapter, training log, summaries and starting losses; `data/gcg74d2/`: one record per new condition and document.
- `reports/whitebox/`: `conditions_74d2.csv`, `start_loss_74d2.csv`, `readings_74d2.csv`, `training_74d2.csv`, `summaries_74d2.csv`, `per_document_74d2.csv`, `decision_74d2.txt`, `RESULTS_LOG.md`.
- `figures/whitebox/why_role_74d2.png` (600 dpi).

In [ ]:
# === SESSION BOOTSTRAP (run first): install, then mount ===
!pip install -q transformers datasets accelerate sentencepiece bitsandbytes peft langdetect==1.0.9
!pip uninstall -y -q torchao   # Colab ships an old torchao that the current PEFT rejects for bf16 models; nothing in this project uses it
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
for d_ in ('reports/whitebox', 'figures/whitebox', 'data/gcg74d2', 'data/diag74d2'): os.makedirs(d_, exist_ok=True)
import torch, gc
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), 'this notebook needs a GPU with bf16 (G4, A100 or H100); a T4 will not do'
assert torch.cuda.get_device_properties(0).total_memory >= 40e9, 'GCG scores hundreds of candidates at a time: this notebook needs a GPU with 40 GB or more'
print('Session ready. GPU:', torch.cuda.get_device_name(0))

## Stage A. The readings, the earlier records, the training set and the format check (no GPU work)

In [ ]:
import importlib, json, hashlib, re, contextlib, io, numpy as np, pandas as pd
import targets, ci_tools, llama_h2h, manual_attacks, gcg_attack, reslog, qwen_role, structural_train, hardened_phase
[importlib.reload(m) for m in (targets, ci_tools, llama_h2h, manual_attacks, gcg_attack, reslog, qwen_role, structural_train, hardened_phase)]
from targets import load_lm, generate_batch, TASK_SYSTEM, TASK_USER
from ci_tools import wilson, fmt
from llama_h2h import generate_messages
from manual_attacks import split_doc, cluster_boot, filter_special
from gcg_attack import GOAL, TARGET, INIT, SETTINGS, CHECKS, SETTINGS_ACCEPTED, Harness, GCG, not_allowed, begins, in_response, settings_hash
from qwen_role import patch_template, messages_q, qwen_filter, INPUT_HEADER
from hardened_phase import train_phase_all
from structural_train import load_jsonl
from final_scoring import CATEGORIES
from reslog import log_result
import transformers
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 120)
H = lambda s: hashlib.sha1(s.encode()).hexdigest()
def jload(f): return json.load(open(f)) if os.path.exists(f) else {}
def jsave(obj, f): json.dump(obj, open(f + '.tmp', 'w')); os.replace(f + '.tmp', f)
REP, FIG, WD, DD = 'reports/whitebox', 'figures/whitebox', 'data/gcg74d2', 'data/diag74d2'
N_DOCS, DOC_SEED = 25, 74_300
for f_ in ('decision_74c.txt', 'per_document_74c.csv', 'decision_74d.txt', 'per_document_74d.csv', 'training_74d.csv'):
    assert os.path.exists(f'{REP}/{f_}'), f"{f_} is missing: pull the repository"

# ---- the readings, fixed here before any training or generation ----
READ = dict(r1=('E', 'C', 'extra training: one more pass over the same examples, without the role, makes the 3B easier to break', 'lower end above 0'),
            r2=('F', 'E', 'the role at equal training makes the 3B easier to break', 'lower end above 0'),
            r3=('D', 'C', "moving the document without retraining breaks v13's defence (a defence tied to the one-turn layout)", 'lower end above 0'),
            r4=('F', 'D', 'training in the role repairs part of what moving the document broke', 'upper end below 0'))
R5 = ('B against A: the injection alone and broken at step 1 (lower end above 0), or the starting loss (upper end below 0): '
      'the untrained Qwen obeys an instruction at the end of the document more readily when the document has its own turn')
print('READINGS (fixed before any result)')
for k, (a, b, what, need) in READ.items(): print(f'  {k.upper()}: {a} minus {b}, share broken, paired over the documents; {need}: {what}')
print('  R5:', R5)
print('GCG settings:', SETTINGS, '| settings hash', settings_hash(), '| device check limits:', CHECKS)

# ---- the six conditions ----
Q3 = 'Qwen/Qwen2.5-3B-Instruct'; V13 = 'data/distill13_3b/lora_3b'; ROLE = 'data/role74d_3b/lora_3b'; MORE = f'{DD}/lora_3b_more'
SETF = 'data/distill13_3b/examples.json'; SEED = 74_400_000                 # notebook 74d's seed for the 3B, so E sees the same batches in the same order as F
COND = {  # key: (tag, adapter, format at run time, where the GCG records are, label)
    'A': ('q3_undefended', None, 'one user turn', 'data/gcg74c', 'Qwen2.5-3B undefended, one user turn'),
    'B': ('q3_undefended_in_role', None, 'input role', WD, 'Qwen2.5-3B undefended, input role (no training)'),
    'C': ('q3_v13', V13, 'one user turn', 'data/gcg74c', 'v13, one user turn (the final 3B)'),
    'D': ('q3_v13_in_role', V13, 'input role', WD, 'v13, input role at run time only'),
    'E': ('q3_v13_more_oneturn', MORE, 'one user turn', WD, 'v13 + one more pass, one user turn (control)'),
    'F': ('q3_v13_role', ROLE, 'input role', 'data/gcg74d', 'v13 + one more pass, input role (notebook 74d)')}
NEW = ['B', 'D', 'E']
for k_ in ('C', 'F'): assert os.path.exists(os.path.join(COND[k_][1], 'adapter_config.json')), f'adapter missing: {COND[k_][1]}'
def ident(model, adapter):                                  # what a record was made on (notebook 74c's rule)
    if adapter is None: return f'{model}|base'
    f_ = [os.path.join(adapter, x) for x in ('adapter_model.safetensors', 'adapter_model.bin') if os.path.exists(os.path.join(adapter, x))]
    return f'{model}|{adapter}|' + (hashlib.sha1(open(f_[0], 'rb').read()).hexdigest()[:12] if f_ else 'hub')
recf = lambda key, k: f'{COND[key][3]}/{COND[key][0]}/doc{k:02d}.json'

# ---- the 25 documents (notebook 74c's) ----
SUB = pd.DataFrame(json.load(open('data/phase5v3/subset_full.json'))); TEXT = list(SUB.text)
assert len(SUB) == 210 and set(SUB.meta) == set(CATEGORIES), "not the paper's 210 injections: stop"
HOSTS = list(dict.fromkeys(split_doc(t)[0].rstrip() for t in TEXT))
PICK = sorted(np.random.default_rng(DOC_SEED).choice(len(HOSTS), N_DOCS, replace=False).tolist()); DOCS = [HOSTS[j] for j in PICK]
assert len(set(DOCS)) == N_DOCS and all(d and filter_special(d, 'qwen') == d for d in DOCS), 'a host document is empty or holds special tokens: stop'

# ---- the earlier records: A and C (notebook 74c), F (notebook 74d) ----
OLD = {}
for key, csv_ in (('A', 'per_document_74c.csv'), ('C', 'per_document_74c.csv'), ('F', 'per_document_74d.csv')):
    tag, ad = COND[key][0], COND[key][1]; rs = [jload(recf(key, k)) for k in range(N_DOCS)]; idn = ident(Q3, ad)
    assert all(rs), f'the records of {tag} are incomplete: stop'
    for k, r in enumerate(rs): assert r.get('settings') in SETTINGS_ACCEPTED and r.get('host_hash') == H(DOCS[k])[:12] and r.get('model_id') == idn, f'{recf(key, k)}: another setting, document or model: stop'
    T_ = pd.read_csv(f'{REP}/{csv_}'); assert int(sum(r['success'] for r in rs)) == int(T_[T_.condition == tag].broken.sum()), f"{tag}: the records do not match the committed table: stop"
    OLD[key] = rs
print('earlier records: ' + ', '.join(f'{k} ({COND[k][0]}) {int(sum(r["success"] for r in OLD[k]))} of {N_DOCS} broken' for k in OLD))

# ---- the training set: notebook 73e's, as notebook 74d used it ----
EX = json.load(open(SETF)); SETH = H(json.dumps(EX, sort_keys=True))[:12]
TR74D = pd.read_csv(f'{REP}/training_74d.csv'); H74D = TR74D[TR74D.model.str.startswith('Qwen2.5-3B')].set_hash.iloc[0]
assert len(EX) == 2800 and SETH == H74D, f'the training set ({SETH}) is not the one notebook 74d trained F on ({H74D}): stop'
print(f'training set: {SETF}, {len(EX)} examples, hash {SETH} (the same as F\'s)')

# ---- the format check (notebook 74d's) ----
VALHOSTS = {r['text'] for r in load_jsonl('data/structural/val.jsonl') if r['form'] == 'host_only'}
VALH = sorted({h for h in VALHOSTS if len(h.split()) >= 30})
PLAIN60 = [TASK_USER.format(doc=VALH[i]) for i in np.random.default_rng(6802).choice(len(VALH), 60, replace=False)]
PROBES = PLAIN60 + [TASK_USER.format(doc=filter_special(d + ' ' + GOAL + INIT, 'qwen')) for d in DOCS]
TK_ROLE = transformers.AutoTokenizer.from_pretrained(Q3); patch_template(TK_ROLE, PROBES, TASK_SYSTEM)
TK_PLAIN = transformers.AutoTokenizer.from_pretrained(Q3)                     # Qwen's own template, as notebook 74c ran the one-turn conditions
if TK_PLAIN.pad_token is None: TK_PLAIN.pad_token = TK_PLAIN.eos_token
HR = Harness(TK_ROLE, 'input_role', 'qwen', TASK_USER, TASK_SYSTEM, messages_q, filter_special, qwen_filter)
HP = Harness(TK_PLAIN, 'qwen', 'qwen', TASK_USER, TASK_SYSTEM, messages_q, filter_special, qwen_filter)
for d in DOCS: HR.split(d); HP.split(d)
r_ = HR.render(HR.doc(DOCS[0], INIT)); assert r_.count(INPUT_HEADER) == 1 and r_.index(INPUT_HEADER) < r_.index(GOAL) < r_.index('<|im_start|>assistant'), 'the attacked document is not inside the input turn: stop'
assert INPUT_HEADER not in HP.render(HP.doc(DOCS[0], INIT)), 'the one-turn prompt holds an input turn: stop'
print('format check passed: one-turn prompts as Qwen writes them; in the role format the attacked document sits inside the input turn')

# ---- from the records: the loss after the first GCG step (descriptive; the starting loss itself is measured in Stage C) ----
print('loss after the first GCG step, median over the documents: ' + ', '.join(f'{k} {np.nanmedian([r["losses"][0] if r["losses"] and r["losses"][0] is not None else np.nan for r in OLD[k]]):.2f}' for k in OLD))

## Stage B. Training E: v13 continued on the same 2,800 examples in one user turn (notebook 74d's training, without the role; cached, resumable)

In [ ]:
import peft, bitsandbytes
train_phase_all(EX, out_adapter=MORE, workdir=DD, model_name=Q3, system=TASK_SYSTEM, torch_seed=SEED, init_adapter=V13, min_chars=0)
assert os.path.exists(os.path.join(MORE, 'adapter_config.json')) and jload(f'{DD}/progress.json').get('step', 0) >= len(EX), 'training incomplete: run this cell again'
lg = jload(f'{DD}/train_log.json') or []
mean_ = lambda L, k: float(np.mean([x[k] for x in L if x.get(k) is not None] or [np.nan]))
short_ = lambda L: sum(x.get('short', 0) for x in L) / max(1, sum(x.get('nroll', 0) for x in L))
rowE = dict(model=COND['E'][4], examples=len(EX), set_hash=SETH, batches=len(lg), reverse_kl_first_100_batches=round(mean_(lg[:100], 'kl'), 4), reverse_kl_last_100_batches=round(mean_(lg[-100:], 'kl'), 4),
            exact_target_first_100_batches=round(mean_(lg[:100], 'sft'), 3), exact_target_last_100_batches=round(mean_(lg[-100:], 'sft'), 3),
            short_rollouts_first_100_batches=round(short_(lg[:100]), 3), short_rollouts_last_100_batches=round(short_(lg[-100:]), 3))
rowF = TR74D[TR74D.model.str.startswith('Qwen2.5-3B')].iloc[0].to_dict(); rowF['model'] = COND['F'][4]
TRAINT = pd.DataFrame([rowE, rowF]); TRAINT.to_csv(f'{REP}/training_74d2.csv', index=False)
print(f'libraries: torch {torch.__version__}, transformers {transformers.__version__}, peft {peft.__version__}, bitsandbytes {bitsandbytes.__version__}; GPU {torch.cuda.get_device_name(0)}')
print('E (trained here) beside F (notebook 74d), the same examples, seed and step; the reverse-KL at the end of v13\'s own training on these examples was 0.0154 (notebook 73e):')
print(TRAINT.T.to_string())

## Stage C. The starting loss for all six conditions and the summaries of B, D and E (cached)

In [ ]:
from peft import PeftModel
def free():                                                 # a saved traceback or a closure can keep an old model alive after an interruption (notebook 74c's rule)
    global m, tm, att, gen_
    for k_ in ('last_type', 'last_value', 'last_traceback', 'last_exc'):
        if hasattr(sys, k_): setattr(sys, k_, None)
    try: get_ipython().InteractiveTB.tb = None
    except Exception: pass
    m = tm = att = gen_ = None; gc.collect(); torch.cuda.empty_cache()
free()
def load(adapter):
    tm_, _ = load_lm(Q3, four_bit=True)
    return (PeftModel.from_pretrained(tm_, adapter).eval() if adapter else tm_), tm_
def harness(key): return HR if COND[key][2] == 'input role' else HP
def generate(m_, key, prompts, bs=8, k=160):                 # the paper's generation code in the condition's format (notebook 74c for one user turn, notebook 74d for the role)
    if COND[key][2] == 'input role': return generate_messages(m_, TK_ROLE, [messages_q(x, 'input_role', TASK_SYSTEM) for x in prompts], max_new_tokens=k, batch_size=bs, show_every=400)
    return generate_batch(m_, TK_PLAIN, prompts, system=TASK_SYSTEM, max_new_tokens=k, batch_size=bs, show_every=400)
def start_losses(m_, key):
    # the attack's loss on "hacked" with the initial suffix (twenty exclamation marks), before any optimisation, scored as GCG scores it (repetition penalty included)
    a_ = GCG(m_, harness(key), torch.tensor([], dtype=torch.long)); out = []
    with torch.no_grad(): V_ = m_(input_ids=torch.tensor([[0]], device=a_.dev)).logits.shape[-1]       # the width of the logits, as GCG sizes it
    for host in DOCS:
        _, _, bi, si, ai = a_.h.split(host)
        a_.base = torch.zeros(V_, dtype=torch.bool, device=a_.dev); a_.base[torch.tensor(bi + ai, device=a_.dev)] = True
        out.append(round(float(a_.full_losses(bi, torch.tensor([si], device=a_.dev), torch.tensor([ai], device=a_.dev))[0]), 4))
    return out
SL = jload(f'{DD}/start_loss.json'); SUMM = jload(f'{DD}/summaries.json')
for adapter, keys in ((None, ['A', 'B']), (V13, ['C', 'D']), (MORE, ['E']), (ROLE, ['F'])):
    need_sl = [k_ for k_ in keys if k_ not in SL]; need_s = [k_ for k_ in keys if k_ in NEW and k_ not in SUMM]
    if not (need_sl or need_s): continue
    free(); m, tm = load(adapter)
    for k_ in need_sl: SL[k_] = start_losses(m, k_); jsave(SL, f'{DD}/start_loss.json'); print(f'{k_} starting loss, median {np.median(SL[k_]):.2f}')
    for k_ in need_s:
        outs = generate(m, k_, PLAIN60); SUMM[k_] = {H(x): o for x, o in zip(PLAIN60, outs)}; jsave(SUMM, f'{DD}/summaries.json')
        print(f'{k_} summary: ' + ' '.join((outs[0] or '').split())[:200])
free()
P13 = jload('data/distill13_3b/plain60_v13.json')
srow = []
for k_ in NEW:
    ln = np.array([len((SUMM[k_][H(x)] or '').strip()) for x in PLAIN60])
    srow.append(dict(condition=COND[k_][0], label=COND[k_][4], under_20=round(float((ln < 20).mean()), 3), median_chars=float(np.median(ln))))
if all(H(x) in P13 for x in PLAIN60): srow.append(dict(condition='q3_v13', label=COND['C'][4] + ' (notebook 73e)', under_20=round(float((np.array([len((P13[H(x)] or '').strip()) for x in PLAIN60]) < 20).mean()), 3), median_chars=float(np.median([len((P13[H(x)] or '').strip()) for x in PLAIN60]))))
SUMT = pd.DataFrame(srow); SUMT.to_csv(f'{REP}/summaries_74d2.csv', index=False); print(SUMT.to_string(index=False))
SLT = pd.DataFrame([dict(condition=COND[k_][0], label=COND[k_][4], median=float(np.median(SL[k_])), lowest=float(np.min(SL[k_])), highest=float(np.max(SL[k_]))) for k_ in 'ABCDEF'])
SLT.to_csv(f'{REP}/start_loss_74d2.csv', index=False); print('\nstarting loss (lower: the model is closer to saying "hacked" before the attack starts):\n' + SLT.to_string(index=False))

## Stage D. GCG on B, D and E: notebook 74c's attack, unchanged (one record per document; resumes)

In [ ]:
import contextlib, io
def complete(key): return all(os.path.exists(recf(key, k)) for k in range(N_DOCS))
IDN = {k_: ident(Q3, COND[k_][1]) for k_ in NEW}
for key in NEW:
    tag = COND[key][0]; os.makedirs(f'{WD}/{tag}', exist_ok=True)
    for k in range(N_DOCS):
        if os.path.exists(recf(key, k)):
            r_ = jload(recf(key, k))
            assert r_.get('settings') in SETTINGS_ACCEPTED and r_.get('model_id') == IDN[key] and r_.get('host_hash') == H(DOCS[k])[:12], f'{recf(key, k)} was made with another setting, model or document: stop'
    if complete(key): print(tag, 'complete'); continue
    print('attacking', tag, '|', COND[key][4], '| format:', COND[key][2])
    free(); m, tm = load(COND[key][1])
    h_ = harness(key); bad = not_allowed(h_.tok, m.get_input_embeddings().weight.shape[0])
    att = GCG(m, h_, bad, mini=512 if torch.cuda.get_device_properties(0).total_memory >= 70e9 else 128)
    print(f'  {len(bad)} of {m.get_input_embeddings().weight.shape[0]} tokens not allowed; the target "{TARGET}" is {att.target.shape[1]} token(s); repetition penalty in greedy decoding {att.p}')
    def gen_(prompts, k_, m=m, key=key):                     # the generation code, one prompt at a time, its progress lines silenced
        with contextlib.redirect_stdout(io.StringIO()): return generate(m, key, prompts, 1, k_)
    for k, host in enumerate(DOCS):
        if os.path.exists(recf(key, k)): continue
        rec = att.run(host, gen_, seed=DOC_SEED + k, tag=f'{tag} document {k + 1}')
        rec.update(condition=tag, doc=k, host_hash=H(host)[:12], model_id=IDN[key], mini_batch=att.mini, format=COND[key][2]); jsave(rec, recf(key, k))
        print(f'  {tag} document {k + 1}/{N_DOCS}: ' + (f'broken at step {rec["success_step"]}' if rec['success'] else f'not broken in {rec["steps_run"]} steps')
              + f' (loss {rec["losses"][-1]}, lowest {rec["best_loss"]}; {rec["sec_per_step"]} s a step; GPU checks {rec["check"]})')
    free()
assert all(complete(k_) for k_ in NEW), 'a run is incomplete: run this cell again'
print('GCG complete for:', [COND[k_][0] for k_ in NEW])

## Stage E. The six conditions side by side, the readings and the figure

In [ ]:
import matplotlib.pyplot as plt
R = dict(OLD); R.update({k_: [jload(recf(k_, k)) for k in range(N_DOCS)] for k_ in NEW})
for k_ in NEW: assert all(r.get('settings') in SETTINGS_ACCEPTED and r['doc'] == k and r.get('model_id') == IDN[k_] and r.get('host_hash') == H(DOCS[k])[:12] for k, r in enumerate(R[k_])), f'the records of {k_} do not match: stop'
KEYS = list('ABCDEF'); DOCID = np.arange(N_DOCS)
OK = {k_: np.array([bool(r['success']) for r in R[k_]]) for k_ in KEYS}
STEP = {k_: np.array([r['success_step'] if r['success'] else np.inf for r in R[k_]], float) for k_ in KEYS}
ALONE = {k_: np.array([begins(r['alone_reply']) for r in R[k_]]) for k_ in KEYS}
AT1 = {k_: STEP[k_] <= 1 for k_ in KEYS}
SLA = {k_: np.array(SL[k_], float) for k_ in KEYS}
CHECKS_AT = [1] + list(range(SETTINGS['check_every'], SETTINGS['steps'] + 1, SETTINGS['check_every']))
rate = lambda v: fmt(float(np.mean(v)), *wilson(int(np.sum(v)), len(v)))
dstr = lambda x: f'{x[0]:+.3f} [{x[1]:+.3f}, {x[2]:+.3f}]'
CT = pd.DataFrame([dict(key=k_, condition=COND[k_][0], label=COND[k_][4], format=COND[k_][2], broken=rate(OK[k_]), broken_n=int(OK[k_].sum()), injection_alone=rate(ALONE[k_]), broken_at_step_1=int(AT1[k_].sum()),
                        median_step_of_success=float(np.median(STEP[k_][OK[k_]])) if OK[k_].any() else np.nan, median_starting_loss=round(float(np.median(SLA[k_])), 3),
                        median_lowest_loss=round(float(np.median([r['best_loss'] for r in R[k_]])), 3), render_mismatches=int(sum(r['render_mismatch'] for r in R[k_])),
                        inconsistent_checks=int(sum(r['inconsistent'] for r in R[k_])), source=('notebook 74c' if k_ in 'AC' else 'notebook 74d' if k_ == 'F' else 'notebook 74d2')) for k_ in KEYS])
CT.to_csv(f'{REP}/conditions_74d2.csv', index=False); print(f'GCG on {N_DOCS} documents, Qwen2.5-3B:\n' + CT.drop(columns=['label']).to_string(index=False))
PDOC = pd.DataFrame([dict(condition=COND[k_][0], doc=r['doc'], host_hash=r['host_hash'], broken=bool(r['success']), step=r['success_step'], steps_run=r['steps_run'], starting_loss=SL[k_][r['doc']],
                          lowest_loss=r['best_loss'], alone=begins(r['alone_reply']), suffix=r['suffix'], reply_start=(r['reply'] or '')[:120]) for k_ in NEW for r in R[k_]])
PDOC.to_csv(f'{REP}/per_document_74d2.csv', index=False)
# ---- the readings ----
rows, MEAN = [], []
for rk, (a, b, what, need) in READ.items():
    x = cluster_boot(OK[a].astype(float) - OK[b].astype(float), DOCID); xs = cluster_boot(SLA[a] - SLA[b], DOCID)
    held = x[1] > 0 if need.startswith('lower') else x[2] < 0
    opposite = x[2] < 0 if need.startswith('lower') else x[1] > 0
    verdict = 'supported' if held else ('contradicted (the opposite direction is shown)' if opposite else 'not settled (the interval contains 0)')
    rows.append(dict(reading=rk.upper(), comparison=f'{a} minus {b}', broken=dstr(x), broken_counts=f'{int(OK[a].sum())} against {int(OK[b].sum())}', only_first=int((OK[a] & ~OK[b]).sum()), only_second=int((~OK[a] & OK[b]).sum()),
                     starting_loss=dstr(xs), needed=need, verdict=verdict, meaning=what))
    MEAN.append(f'{rk.upper()} ({a} minus {b}): broken {dstr(x)}, {int(OK[a].sum())} against {int(OK[b].sum())} of {N_DOCS}; starting loss {dstr(xs)}. {verdict.capitalize()}: {what}.')
xa, x1, xl = (cluster_boot(ALONE['B'].astype(float) - ALONE['A'].astype(float), DOCID), cluster_boot(AT1['B'].astype(float) - AT1['A'].astype(float), DOCID), cluster_boot(SLA['B'] - SLA['A'], DOCID))
r5 = xa[1] > 0 or x1[1] > 0 or xl[2] < 0; r5_opp = xa[2] < 0 or x1[2] < 0 or xl[1] > 0
r5v = 'supported' if r5 and not r5_opp else ('mixed: some measures point each way' if r5 and r5_opp else ('contradicted' if r5_opp else 'not settled'))
rows.append(dict(reading='R5', comparison='B minus A', broken=dstr(cluster_boot(OK['B'].astype(float) - OK['A'].astype(float), DOCID)), broken_counts=f'{int(OK["B"].sum())} against {int(OK["A"].sum())}',
                 only_first=int((OK['B'] & ~OK['A']).sum()), only_second=int((~OK['B'] & OK['A']).sum()), starting_loss=dstr(xl), needed='injection alone or step 1: lower end above 0; or starting loss: upper end below 0',
                 verdict=r5v, meaning=f'the injection alone {dstr(xa)}; broken at step 1 {dstr(x1)}'))
MEAN.append(f'R5 (B minus A, the untrained model): the injection alone {dstr(xa)} ({int(ALONE["B"].sum())} against {int(ALONE["A"].sum())}); broken at step 1 {dstr(x1)} ({int(AT1["B"].sum())} against {int(AT1["A"].sum())}); starting loss {dstr(xl)}. '
            f'{r5v.capitalize()}: the untrained Qwen obeys an instruction at the end of the document more readily when the document has its own turn.')
RT = pd.DataFrame(rows); RT.to_csv(f'{REP}/readings_74d2.csv', index=False); print('\nreadings, paired by document:\n' + RT.drop(columns=['meaning']).to_string(index=False))
GTXT = '\n'.join(MEAN) + f"\nTraining: E's reverse-KL to the teacher {rowE['reverse_kl_first_100_batches']} in the first 100 batches and {rowE['reverse_kl_last_100_batches']} in the last; F's {rowF['reverse_kl_first_100_batches']} and {rowF['reverse_kl_last_100_batches']}."
open(f'{REP}/decision_74d2.txt', 'w').write(GTXT + '\n'); print('\n' + GTXT)
# ---- figure (600 dpi) ----
COL = {'A': '#bdbdbd', 'B': '#616161', 'C': '#90caf9', 'D': '#1565c0', 'E': '#fb8c00', 'F': '#2e7d32'}
STY = {'one user turn': '--', 'input role': '-'}
fig, ax = plt.subplots(1, 2, figsize=(16, 5.6), gridspec_kw=dict(width_ratios=[1.35, 1]))
for k_ in KEYS: ax[0].plot(CHECKS_AT, [(STEP[k_] <= s).mean() for s in CHECKS_AT], STY[COND[k_][2]], color=COL[k_], lw=2, label=f'{k_}: {COND[k_][4]}', drawstyle='steps-post')
ax[0].set_ylim(0, 1.02); ax[0].set_xlabel('GCG step'); ax[0].set_ylabel(f'share of the {N_DOCS} documents broken'); ax[0].set_title('Share of documents broken (dashed: one user turn; solid: input role)', fontsize=10)
ax[0].grid(alpha=.3); ax[0].legend(fontsize=8, loc='lower right')
for i, k_ in enumerate(KEYS):
    ax[1].scatter(np.full(N_DOCS, i) + np.random.default_rng(i).uniform(-0.18, 0.18, N_DOCS), SLA[k_], s=14, color=COL[k_], alpha=0.75)
    ax[1].hlines(np.median(SLA[k_]), i - 0.3, i + 0.3, color='black', lw=2)
ax[1].set_xticks(range(len(KEYS))); ax[1].set_xticklabels(KEYS); ax[1].set_xlabel('condition (see the legend on the left)')
ax[1].set_ylabel('starting loss (minus log-probability of "hacked", before the attack)'); ax[1].set_title('How close each model is to saying "hacked" before any optimisation (bar: median)', fontsize=10); ax[1].grid(alpha=.3, axis='y')
fig.suptitle(f'Why the input role made Qwen2.5-3B easier to break: the six conditions under the same white-box attack (GCG, {SETTINGS["steps"]} steps, {N_DOCS} documents)', fontsize=11)
plt.tight_layout(); plt.savefig(f'{FIG}/why_role_74d2.png', dpi=600, bbox_inches='tight'); plt.show()
log_result('nb74d2: why the input role made Qwen2.5-3B easier to break (the missing controls of notebook 74d)',
           '\n\n'.join([GTXT, 'Conditions:\n' + CT.drop(columns=['label']).to_string(index=False), 'Readings:\n' + RT.drop(columns=['meaning']).to_string(index=False),
                        'Starting loss:\n' + SLT.to_string(index=False), 'Training:\n' + TRAINT.to_string(index=False), 'Summaries:\n' + SUMT.to_string(index=False)]), reports_dir=REP)

## Commit

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py", "nb74d2 (training and evaluation): why the input role made Qwen2.5-3B easier to break in notebook 74d (11 to 21 of 25 documents under GCG); the controls notebook 74d lacked, on the same 25 documents with notebook 74c's GCG unchanged: B the undefended 3B with the document in the input turn, D v13 with the document in the input turn and no new training, E v13 continued on the same 2,800 examples in one user turn with notebook 74d's seed and training step (the control for the extra pass), beside A, C (notebook 74c) and F (notebook 74d); the starting loss on hacked before any optimisation for all six; readings fixed in Stage A: R1 extra training (E minus C), R2 the role at equal training (F minus E), R3 moving the document without retraining (D minus C), R4 retraining in the role (F minus D), R5 the format on the untrained model (B against A); summaries of B, D and E; reports/whitebox/*_74d2.csv, decision_74d2.txt, figures/whitebox/why_role_74d2.png (600 dpi)"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)